# Step 3. Adaptation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/younesStrittmatter/sweetpea-starter/blob/main/tutorial/notebooks/3_adaptation.ipynb)

The tutorial step without an assistant. Run the cells from top to bottom; the tables below are from one run, yours will differ where the design leaves things open.

In [ ]:
%pip install -q sweetpea pandas

**So far.** Step 1 chose how many copies of each pair a block holds: 2 of each congruent pair, 1 of each incongruent pair. This step turns that choice into the manipulation.

**The question.** Do people adjust to how frequent conflict is?

**Prediction.** The congruency effect is smaller in mostly incongruent blocks than in mostly congruent blocks.

**The alternative to rule out.** In a mostly congruent block the word predicts the ink, so people can simply learn to follow the word.

**The analysis.** The congruency effect in mostly congruent blocks against the congruency effect in mostly incongruent blocks.

### As in step 1

In [ ]:
import pandas as pd
from sweetpea import (Factor, DerivedLevel, WithinTrial, Transition, Exclude, MinimumTrials,
                      CrossBlock, synthesize_trials, CMSGen)

colours = ["red", "green", "blue"]

# regular factors: set freely on every trial
colour = Factor("colour", colours)      # the ink
word = Factor("word", colours)          # what the word says

In [ ]:
# a derived factor: it follows from colour and word on the same trial
def same(colour, word):
    return colour == word


def different(colour, word):
    return colour != word


congruency = Factor("congruency", [
    DerivedLevel("congruent", WithinTrial(same, [colour, word])),
    DerivedLevel("incongruent", WithinTrial(different, [colour, word])),
])

In [ ]:
def pairs_with_copies(congruent_copies, incongruent_copies):
    """Two helper factors that make every congruent pair occur `congruent_copies` times
    and every incongruent pair `incongruent_copies` times."""
    copy = Factor("copy", [str(n) for n in range(1, max(congruent_copies, incongruent_copies) + 1)])

    def too_many(copy, colour, word):
        return int(copy) > (congruent_copies if colour == word else incongruent_copies)

    def wanted(copy, colour, word):
        return not too_many(copy, colour, word)

    extra_copy = Factor("extra_copy", [
        DerivedLevel("too many", WithinTrial(too_many, [copy, colour, word])),
        DerivedLevel("wanted", WithinTrial(wanted, [copy, colour, word])),
    ])
    return copy, extra_copy

### New: two kinds of block

The number of copies sets the proportion. 6 copies of each congruent pair and 1 of each incongruent pair is 18 of 24 trials congruent; 2 and 3 copies is 6 of 24.

In [ ]:
COPIES = {"mostly congruent": (6, 1), "mostly incongruent": (2, 3)}     # copies of each (congruent, incongruent) pair


def block(context):
    copy, extra_copy = pairs_with_copies(*COPIES[context])
    return CrossBlock(design=[colour, word, congruency, copy, extra_copy],
                      crossing=[colour, word, copy],
                      constraints=[Exclude((extra_copy, "too many"))],
                      require_complete_crossing=False)

### A session of four blocks

Each block is generated on its own and gets its context as a column.

In [ ]:
blocks = []
for number, context in enumerate(["mostly congruent", "mostly incongruent", "mostly incongruent", "mostly congruent"], start=1):
    trials = pd.DataFrame(synthesize_trials(block(context), 1, CMSGen)[0])
    trials.insert(0, "context", context)
    trials.insert(0, "block", number)
    blocks.append(trials)
session = pd.concat(blocks, ignore_index=True)
display(pd.crosstab(session["context"], session["congruency"]))

congruency,congruent,incongruent
context,,
mostly congruent,36,12
mostly incongruent,12,36


### How well does the word tell the ink?

For each word, the share of trials on which it is shown in its own colour. That number is the alternative explanation.

In [ ]:
session["word_is_ink"] = session["word"] == session["colour"]
display(session.groupby(["context", "word"])["word_is_ink"].mean().round(2).unstack())

word,blue,green,red
context,,,
mostly congruent,0.75,0.75,0.75
mostly incongruent,0.25,0.25,0.25


**Check.** Are the proportions what was asked for? How many incongruent trials does a mostly congruent block have: is that enough to measure the effect there? A word is in its own colour on 75% of the trials in one context and 25% in the other: how could learning and control be told apart?

**Try.** Change the copies to `(4, 1)` and `(1, 2)`. What are the proportions and the block lengths now?

**Next.** Step 4 asks whether this adjustment has a price when the task changes.